# 01 — FPL Data Collection

Collects raw data from the FPL official API and saves it as Parquet files.

**Endpoints used:**
- `/bootstrap-static/` — all players, teams, gameweek info
- `/fixtures/` — all fixtures + fixture difficulty ratings (FDR)
- `/element-summary/{id}/` — per-player gameweek history

**Outputs:**
- `data/raw/fpl_players.parquet` — one row per player, current season snapshot
- `data/raw/fpl_gameweeks.parquet` — one row per player per gameweek
- `data/raw/fpl_fixtures.parquet` — all fixtures with FDR
- `data/raw/fpl_teams.parquet` — team ID to name mapping

## 0. Smoke test — verify API is reachable

In [ ]:
import requests

BASE_URL = "https://fantasy.premierleague.com/api/"

r = requests.get(f"{BASE_URL}bootstrap-static/")
r.raise_for_status()
data = r.json()

print("Status:", r.status_code)
print("Top-level keys:", list(data.keys()))
print("Player count:", len(data["elements"]))
print("Team count:", len(data["teams"]))
print("Total gameweeks:", len(data["events"]))

## 1. Imports & setup

In [ ]:
import time
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
RAW.mkdir(parents=True, exist_ok=True)

## 2. Bootstrap — players, teams, gameweek metadata

In [ ]:
bootstrap = requests.get(f"{BASE_URL}bootstrap-static/").json()

# --- Players (elements) ---
players_df = pd.DataFrame(bootstrap["elements"])
print(players_df.shape)
players_df.head()

In [ ]:
# Columns we care about at this stage
PLAYER_COLS = [
    "id", "first_name", "second_name", "web_name",
    "element_type",   # 1=GKP, 2=DEF, 3=MID, 4=FWD
    "team",           # team ID
    "now_cost",       # price x 10 (e.g. 65 = £6.5m)
    "selected_by_percent",
    "total_points",
    "minutes",
    "goals_scored", "assists", "clean_sheets", "bonus",
    "ict_index",
    "form",
    "status",         # a=available, d=doubtful, i=injured, s=suspended, u=unavailable
]

players_df = players_df[PLAYER_COLS].copy()
players_df["now_cost"] = players_df["now_cost"] / 10  # convert to £m
players_df.head()

In [ ]:
# --- Teams ---
teams_df = pd.DataFrame(bootstrap["teams"])[["id", "name", "short_name"]]
teams_df.head()

In [ ]:
# --- Gameweek metadata ---
events_df = pd.DataFrame(bootstrap["events"])[
    ["id", "name", "deadline_time", "finished", "is_current", "is_next",
     "average_entry_score", "highest_score"]
]
current_gw = events_df.loc[events_df["is_current"], "id"].values
print("Current gameweek:", current_gw)
events_df.head()

## 3. Fixtures

In [ ]:
fixtures_raw = requests.get(f"{BASE_URL}fixtures/").json()
fixtures_df = pd.DataFrame(fixtures_raw)

FIXTURE_COLS = [
    "id", "event",           # gameweek number
    "team_h", "team_a",
    "team_h_difficulty", "team_a_difficulty",  # FDR (1-5)
    "team_h_score", "team_a_score",
    "finished", "kickoff_time",
]

fixtures_df = fixtures_df[FIXTURE_COLS].copy()
print(fixtures_df.shape)
fixtures_df.head()

## 4. Per-player gameweek history

One request per player — ~700 requests. A small delay is added between calls to avoid hammering the API.

In [ ]:
# WARNING: takes ~5-10 minutes for the full player list. Run once and cache to Parquet.

all_gw_records = []
player_ids = players_df["id"].tolist()

for i, pid in enumerate(player_ids):
    url = f"{BASE_URL}element-summary/{pid}/"
    resp = requests.get(url)
    if resp.status_code != 200:
        print(f"  Skipped player {pid} — status {resp.status_code}")
        continue

    for row in resp.json().get("history", []):
        row["player_id"] = pid
        all_gw_records.append(row)

    if i % 50 == 0:
        print(f"  {i}/{len(player_ids)} players fetched...")

    time.sleep(0.05)  # 50ms between requests

gw_df = pd.DataFrame(all_gw_records)
print("Total gameweek rows:", len(gw_df))
gw_df.head()

In [ ]:
GW_COLS = [
    "player_id", "round",
    "total_points",
    "minutes", "goals_scored", "assists", "clean_sheets",
    "goals_conceded", "own_goals", "penalties_saved", "penalties_missed",
    "yellow_cards", "red_cards", "saves", "bonus", "bps",
    "influence", "creativity", "threat", "ict_index",
    "value",       # price at time of this gameweek (x 10)
    "selected",    # ownership count at time of this gameweek
    "was_home",
    "opponent_team",
    "kickoff_time",
]

gw_df = gw_df[[c for c in GW_COLS if c in gw_df.columns]].copy()
gw_df["value"] = gw_df["value"] / 10
gw_df.head()

## 5. Save to Parquet

In [ ]:
players_df.to_parquet(RAW / "fpl_players.parquet", index=False)
fixtures_df.to_parquet(RAW / "fpl_fixtures.parquet", index=False)
gw_df.to_parquet(RAW / "fpl_gameweeks.parquet", index=False)
teams_df.to_parquet(RAW / "fpl_teams.parquet", index=False)

print("Saved:")
for name, df in [
    ("fpl_players.parquet", players_df),
    ("fpl_fixtures.parquet", fixtures_df),
    ("fpl_gameweeks.parquet", gw_df),
    ("fpl_teams.parquet", teams_df),
]:
    print(f"  {name:<30} {len(df):>5} rows")